In [0]:
from pyspark.sql import functions as F
# Ruta del archivo original en S3
ruta_raw = (
    "s3://stasys-bigdata/"
    "Consulta_C1_Emergencia_2024_1.csv"
)
# Leer el CSV respetando el separador y las comillas
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .option("inferSchema", "false")
    .option("quote", '"')
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .load(ruta_raw)
)
print("LECTURA RAW COMPLETADA")
print("Total de registros:", df_raw.count())
print("Total de columnas:", len(df_raw.columns))
display(df_raw.limit(10))

In [0]:
from pyspark.sql import types as T
from pyspark.sql import functions as F

# 1. Definición del Esquema
schema_c1 = T.StructType([
    T.StructField("ANHO", T.IntegerType(), True),
    T.StructField("MES", T.IntegerType(), True),
    T.StructField("UBIGEO", T.StringType(), True),
    T.StructField("DEPARTAMENTO", T.StringType(), True),
    T.StructField("PROVINCIA", T.StringType(), True),
    T.StructField("DISTRITO", T.StringType(), True),
    T.StructField("SECTOR", T.StringType(), True),
    T.StructField("CATEGORIA", T.StringType(), True),
    T.StructField("CO_IPRESS", T.StringType(), True),
    T.StructField("RAZON_SOC", T.StringType(), True),
    T.StructField("TIPO_SEXO", T.IntegerType(), True),
    T.StructField("GRUPO_EDAD", T.IntegerType(), True),
    T.StructField("NRO_TOTAL_ATENCIONES", T.IntegerType(), True),
    T.StructField("NRO_TOTAL_ATENDIDOS", T.IntegerType(), True)
])

# 2. Rutas de Origen y Destino
nombre_archivo = "Consulta_C1_Emergencia_2024_1.csv"
ruta_raw = f"s3://stasys-bigdata/{nombre_archivo}"
ruta_silver_dir = "s3://stasys-bigdata/silver/"
ruta_temp_dir = "s3://stasys-bigdata/silver/_temp_csv/"
ruta_final_archivo = f"{ruta_silver_dir}{nombre_archivo}"

# 3. Lectura del Archivo RAW
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .schema(schema_c1)
    .option("quote", '"')
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .load(ruta_raw)
)

total_raw = df_raw.count()

# 4. Limpieza y Estandarización
df_prepared = (
    df_raw
    .withColumn("DEPARTAMENTO", F.trim(F.upper(F.col("DEPARTAMENTO"))))
    .withColumn("PROVINCIA", F.trim(F.upper(F.col("PROVINCIA"))))
    .withColumn("DISTRITO", F.trim(F.upper(F.col("DISTRITO"))))
    .withColumn("RAZON_SOC", F.trim(F.col("RAZON_SOC")))
    .withColumn("UBIGEO", F.lpad(F.coalesce(F.col("UBIGEO"), F.lit("")), 6, "0"))
    .withColumn("NRO_TOTAL_ATENCIONES", F.coalesce(F.col("NRO_TOTAL_ATENCIONES"), F.lit(0)))
    .withColumn("NRO_TOTAL_ATENDIDOS", F.coalesce(F.col("NRO_TOTAL_ATENDIDOS"), F.lit(0)))
)

# 5. Eliminación de Duplicados
df_dedup = df_prepared.dropDuplicates()
total_dedup = df_dedup.count()
duplicados_eliminados = total_raw - total_dedup

# 6. Eliminación de Incoherencias y Reglas de Negocio
df_silver = (
    df_dedup
    .filter(
        (F.col("MES").between(1, 12)) & 
        (F.col("TIPO_SEXO").isin([1, 2])) & 
        (F.col("GRUPO_EDAD").between(1, 15)) &
        (F.col("NRO_TOTAL_ATENCIONES") >= F.col("NRO_TOTAL_ATENDIDOS"))
    )
)

total_silver = df_silver.count()
incoherencias_eliminadas = total_dedup - total_silver

# --- IMPRESIÓN DE MÉTRICAS DE CALIDAD ---
print("\n" + "="*45)
print("     MÉTRICAS DE CALIDAD DE DATOS (CAPA SILVER)     ")
print("="*45)
print(f"Total Registros Iniciales (Raw)    : {total_raw}")
print(f"Registros Duplicados Eliminados    : {duplicados_eliminados}")
print(f"Registros Incoherentes Eliminados  : {incoherencias_eliminadas}")
print(f"Total Registros Limpios (Silver)   : {total_silver}")
print(f"Tasa de Retención de Calidad       : {round((total_silver / total_raw) * 100, 2)}%")
print("="*45 + "\n")

# 7. Guardar temporalmente en un solo archivo
(
    df_silver.coalesce(1)
    .write
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .mode("overwrite")
    .save(ruta_temp_dir)
)

# 8. Mover y Renombrar al nombre exacto original dentro de /silver/
# Localizamos el archivo 'part-00000...' generado por Spark
archivo_part = [f.path for f in dbutils.fs.ls(ruta_temp_dir) if f.name.startswith("part-")][0]

# Movemos el archivo a su ubicación y nombre final en S3
dbutils.fs.mv(archivo_part, ruta_final_archivo)

# Limpiamos la carpeta temporal
dbutils.fs.rm(ruta_temp_dir, recurse=True)

print(f"¡Éxito! El archivo limpio se guardó exactamente como:")
print(f"-> {ruta_final_archivo}")

In [0]:
# --- GUARDAR EN DATABRICKS ---

tabla_destino = "big_data_team2.edy.produccion_asistencial"

ruta_raw = (
    "s3://stasys-bigdata/silver/"
    "Consulta_C1_Emergencia_2024_1.csv"
)

print(f"Iniciando la escritura en la tabla: {tabla_destino}...")

(df_raw.write
 .format("delta")  # Delta es el formato estándar y recomendado en Databricks
 .mode("overwrite")  # Usa "overwrite" para borrar lo que haya y meter datos nuevos, o "append" para acumular datos
 .option("mergeSchema", "true")  # Útil si la estructura de columnas cambia en el futuro
 .saveAsTable(tabla_destino))

print("¡ESCRITURA COMPLETADA CON ÉXITO!")